# PCB Router — Round-Robin v2 (MaskablePPO)

Trains the round-robin trace-growth router from
[`adikeshn/pcb-router-world`](https://github.com/adikeshn/pcb-router-world), branch **`round-robin-v2`**.

Workflow (run cells top to bottom):
1. **Setup** — clone the branch, install deps
2. **W&B login**
3. **Hyperparameters** — every tunable in one dict (suggested 6-trace defaults filled in)
4. **Board preview** — *look at the board before training*: pins, connector, obstacles, dimensions, breakout
5. **Acceptance checks** — zero-violation test + reward-scale dominance test (both must PASS)
6. **Train** — MaskablePPO with full W&B metric + board-image logging
7. **Results** — final portfolio boards + metrics

Runtime: CPU is fine (vector obs + MLP). A GPU runtime speeds up PPO updates a little but is not required.


## 1 · Setup


In [ ]:
BRANCH = 'round-robin-v2'
REPO   = 'https://github.com/adikeshn/pcb-router-world.git'

import os, sys
if not os.path.exists('pcb-router-world'):
    !git clone --branch $BRANCH --single-branch $REPO
%cd pcb-router-world
!pip -q install -r requirements.txt
sys.path.insert(0, os.getcwd())
print('setup complete')


## 2 · Weights & Biases login


In [ ]:
import wandb
wandb.login()   # paste your API key when prompted


## 3 · Hyperparameters

Everything below maps 1:1 to `pcb_router_rr2.config.Config`. The values shown are the
**suggested defaults for the 6-trace, 3-stacked-on-3 test board** (120 × 180 mm).
Edit anything; unknown keys raise immediately so typos can't silently no-op.


In [ ]:
HPARAMS = dict(
    # ---------------- board (mm) ----------------
    board_width_mm   = 120.0,
    board_height_mm  = 180.0,
    edge_clearance_mm= 2.0,
    connector_rect   = (82.0, 167.0, 98.0, 180.0),
    pins = [                       # 3-stacked-on-3, 3 mm pitch
        (87.0, 178.5), (90.0, 178.5), (93.0, 178.5),   # top row
        (87.0, 171.7), (90.0, 171.7), (93.0, 171.7),   # bottom row
    ],
    obstacles = [],                # extra keep-outs: [(x0,y0,x1,y1), ...]

    # ---------------- clearances (mm) ----------------
    trace_clearance_mm    = 1.33,  # between different traces (enforced everywhere)
    self_clearance_mm     = 0.60,  # trace vs its own older path
    obstacle_clearance_mm = 1.00,

    # ---------------- growth ----------------
    step_mm       = 1.0,
    budget_min_mm = 25.0,          # growth budget sampled per episode ->
    budget_max_mm = 45.0,          #   total length is a searchable dimension
    self_skip_mm  = 2.0,           # arc-length own-path exemption window
    ban_reverse   = True,

    # ---------------- reward ----------------
    w_spacing_dense    = 0.02,     # per-round hinge on MIN pairwise tip distance
    spacing_target_mm  = 16.0,     # hinge saturation (spec 13 + margin)
    w_edge_penalty     = 0.003,
    edge_soft_mm       = 8.0,
    w_path_penalty     = 0.003,
    path_soft_mm       = 4.0,
    w_terminal_base    = 5.0,      # completion bonus (gated: complete + 0 violations)
    w_terminal_quality = 5.0,      # scales quality blend below
    q_endpoint_spacing = 0.6,
    q_path_clearance   = 0.3,
    q_short_budget     = 0.1,
    endpoint_spec_mm   = 13.0,     # LABEL + portfolio tier only, never gate/reward

    # ---------------- portfolio ----------------
    portfolio_k        = 5,
    min_moved_frac     = 0.5,      # diversity: >=3 of 6 endpoints must shift...
    min_point_shift_mm = 13.0,     # ...by >= this vs every existing entry

    # ---------------- PPO ----------------
    total_timesteps = 2_000_000,   # ~1.5-3 h on a Colab CPU runtime
    n_envs        = 8,
    seed          = 0,
    learning_rate = 3e-4,
    n_steps       = 512,
    batch_size    = 512,
    n_epochs      = 6,
    gamma         = 0.995,         # long episodes (150-270 steps)
    gae_lambda    = 0.95,
    ent_coef      = 0.01,
    clip_range    = 0.2,
    net_arch      = [256, 256],

    # ---------------- exploration / eval / logging ----------------
    explorer_every_episodes = 200, # ForcedExplorer burst cadence
    explorer_episodes       = 5,
    explorer_momentum       = 0.9, # persistent walk (uniform boxes in ~95%)
    eval_every_steps        = 100_000,
    eval_episodes           = 8,   # deterministic, spread across budget range
    render_every_episodes   = 250, # training-board images to W&B
    log_every_episodes      = 10,

    wandb_project  = 'pcb-routing',
    wandb_run_name = None,         # None -> timestamped
    wandb_mode     = 'online',
)

from pcb_router_rr2.config import Config
cfg = Config().override(**HPARAMS)
cfg.validate()
print(f'config OK — {cfg.n_traces} traces, obs/reward/PPO parameters loaded')


## 4 · Board preview — check the layout BEFORE training

Shows board outline with dimensions, edge-clearance zone (dashed), connector,
numbered pin startpoints, any obstacles, and the deterministic breakout
(dotted) ending at the agent hand-off tips (squares).

**Do not start training until this looks like your board.**


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from pcb_router_rr2.board import Board
from pcb_router_rr2.rendering import preview_figure

print(Board.from_config(cfg).summary())
print()
fig = preview_figure(cfg, save_path='board_preview.png')
plt.show()


## 5 · Acceptance checks (run before every training launch)

* **zero_violation_check** — 200 masked-random episodes must audit to **zero**
  clearance violations (the v1 'residual crossings' regression test).
* **reward_scale_check** — max cumulative dense reward must stay well below the
  terminal completion bonus (the v1 reward-imbalance regression test).

If either fails, fix the config — do not train.


In [ ]:
from pcb_router_rr2.validate import run_all
assert run_all(cfg), 'Acceptance checks FAILED — do not train with this config.'


## 6 · Train

W&B panels to watch:
* `train/*` — completion / gate-pass / spec-pass / boxed-in rates, endpoint spacing, length spread
* `reward/*` — per-term decomposition (catch any term silently dominating)
* `eval/*` — deterministic policy across the budget range
* `board/training_episode`, `board/eval_best`, `board/portfolio` — rendered boards

Interrupting the cell (■) is safe: the model and portfolio are saved on KeyboardInterrupt.


In [ ]:
from pcb_router_rr2.train import train

run_dir = train(cfg)
print('artifacts in', run_dir)


## 7 · Results — final portfolio


In [ ]:
import json, glob
from IPython.display import Image, display

with open(f'{run_dir}/portfolio/portfolio.json') as f:
    index = json.load(f)

print(f'{len(index)} portfolio entries (lexicographic: spec-pass first, then terminal reward)')
for e in index:
    print(f"  rank {e['rank']}: spec={'PASS' if e['meets_spec'] else 'miss'}  "
          f"terminal={e['reward_terminal']:.2f}  "
          f"min_spacing={e['min_endpoint_spacing_mm']:.1f}mm  "
          f"budget={e['budget_mm']:.0f}mm  "
          f"len_spread={e['length_spread_mm']:.3f}mm")
    display(Image(e['png'], width=520))


### Optional: resume / extend a run

```python
from pcb_router_rr2.train import train
run_dir = train(cfg, resume_model=f'{run_dir}/model_final.zip')
```

### Optional: download everything
```python
!zip -r results.zip {run_dir}
from google.colab import files; files.download('results.zip')
```
